# Rbio Demo: Dual-Task LLM Post-Training Experiment

<a target="_blank" href="https://colab.research.google.com/github/raphaelrubrice/rbio/blob/main/rbio_colab_dual_experiment.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Estimated Time to Complete:** ~60–90 minutes with A100 GPU (10 training runs total).

**Google Colab Note:** It is strongly recommended to run this notebook with an A100 GPU, which is only included with Google Colab Pro or Enterprise paid services.
Alternatively, a "pay as you go" option is available. See [Colab Service Plans](https://colab.research.google.com/signup) for details.

## Learning Goals

This notebook extends the base Rbio demo by comparing **standard GRPO training** against **dual-task GRPO training**, each run 5 times with different random seeds.

**What you will do:**
1. Set up the full Rbio environment on Colab, including PerturbQA knowledge graphs.
2. Train the LLM **without** the dual task (5 seeds) and collect reward curves + held-out accuracy.
3. Train the LLM **with** the dual task (5 seeds) and collect the same metrics.
4. Visualize results with seaborn and export plots as PNG files.

---

## Pre-requisites
* **GPU**: A100 recommended. T4 is possible with a smaller model.
* **Libraries**: numpy, pandas, torch, datasets, transformers, trl, accelerate, seaborn.

## Introduction

Rbio post-trains an LLM using a soft verification signal from a simplified Virtual Cell Model (VCM). This notebook adds a **dual task**: after predicting whether gene A affects gene B, the model is asked to identify gene B from a candidate list given knowledge-graph context. This second task acts as an additional grounding signal during training.

See the [preprint](https://www.biorxiv.org/content/10.1101/2025.08.18.670981v3) and the [PerturbQA dataset](https://github.com/genentech/PerturbQA) for full details.

# Setup

### Setup Google Colab

Choose the A100 runtime from the "Connect" dropdown in the upper-right corner before running any cells.

## Download necessary assets

In [1]:
!pip install gdown -q

In [2]:
!git clone https://github.com/raphaelrubrice/rbio.git /content/rbio

Cloning into '/content/rbio'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 172 (delta 23), reused 29 (delta 12), pack-reused 124 (from 1)
Receiving objects: 100% (172/172), 7.52 MiB | 27.81 MiB/s, done.
Resolving deltas: 100% (88/88), done.


## Download MLP checkpoint and ESM embeddings

In [3]:
# Download embeddings
!gdown "https://drive.google.com/uc?id=1hBhnanbna2t5TBZlhPr6Y8sPZbOX_BV9"

# Download MLP checkpoint
!gdown "https://drive.google.com/uc?id=1o6FvLUGlFz8f-vYrhc1shAYVxtGNQEkh"

Downloading...
From (original): https://drive.google.com/uc?id=1hBhnanbna2t5TBZlhPr6Y8sPZbOX_BV9
From (redirected): https://drive.google.com/uc?id=1hBhnanbna2t5TBZlhPr6Y8sPZbOX_BV9&confirm=t&uuid=0b8f3592-f9ea-4395-96b1-0ae30d88a983
To: /content/esm_embedding_dictionary_filled.pkl
100% 112M/112M [00:00<00:00, 129MB/s]
Downloading...
From: https://drive.google.com/uc?id=1o6FvLUGlFz8f-vYrhc1shAYVxtGNQEkh
To: /content/mlp_model.pt
100% 1.31M/1.31M [00:00<00:00, 171MB/s]


## Download training data

In [4]:
!gdown "https://drive.google.com/uc?id=16WR4a4bdqiWXd72HToFvAJ66e1jxsp-k"

Downloading...
From: https://drive.google.com/uc?id=16WR4a4bdqiWXd72HToFvAJ66e1jxsp-k
To: /content/k562-train-v0.3.0.csv
100% 81.5M/81.5M [00:01<00:00, 61.9MB/s]


## Download PerturbQA data (KG + gene summaries)

KG and gene summary data are hosted on Google Drive. Downloads ~400 MB total.

In [5]:
!rm -rf /content/rbio/PerturbQA/kg && rm -rf /content/rbio/PerturbQA/gene_summaries

In [6]:
!python /content/rbio/PerturbQA/download_data.py
print("PerturbQA setup complete.")
!ls /content/rbio/PerturbQA

Downloading...
From (original): https://drive.google.com/uc?id=1czfVavbkkZFBNLYMCRIICh2Czr-LOfSU
From (redirected): https://drive.google.com/uc?id=1czfVavbkkZFBNLYMCRIICh2Czr-LOfSU&confirm=t&uuid=74244521-6d79-4167-a72f-93ea560eb54a
To: /content/rbio/PerturbQA/kg.zip
100% 46.1M/46.1M [00:00<00:00, 200MB/s]
  Extracting → /content/rbio/PerturbQA/kg/
Downloading...
From: https://drive.google.com/uc?id=1sdh14LnF3unPWZiTmYYOMN29FH3McIUX
To: /content/rbio/PerturbQA/gene_summary.zip
100% 6.42M/6.42M [00:00<00:00, 195MB/s]
  Extracting → /content/rbio/PerturbQA/gene_summary/
Traceback (most recent call last):
  File "/content/rbio/PerturbQA/download_data.py", line 77, in <module>
    main()
  File "/content/rbio/PerturbQA/download_data.py", line 58, in main
    shutil.move(dest / "perturbqa" / "datasets" / "kg", dest)
  File "/usr/lib/python3.12/shutil.py", line 845, in move
    raise Error("Destination path '%s' already exists" % real_dst)
shutil.Error: Destination path '/content/rbio/Pertur

## Set up directories

## Download PerturbQA data (KG + gene summaries)

Uses `PerturbQA/download_data.py` from the cloned repo to fetch KG and gene summaries
from Google Drive. Downloads ~400 MB total.

## Install dependencies

In [7]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["DISABLE_MLFLOW_INTEGRATION"] = "true"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TRANSFORMERS_IMAGE_TRANSFORMS"] = "0"

In [8]:
!pip install -q uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 111.6 MB/s eta 0:00:00


In [15]:
!uv pip uninstall torchvision torchaudio torch -q
!uv pip install --no-cache-dir --index-url https://download.pytorch.org/whl/cu124 \
  torch==2.6.0 torchaudio==2.6.0 -q
!uv pip install --no-cache-dir \
  transformers==4.51.3 \
  trl==0.14.0 \
  datasets==3.5.0 \
  pandas==2.2.3 "protobuf>=4.25" accelerate tiktoken seaborn -q

In [ ]:
# Restart runtime
import os
os.kill(os.getpid(), 9)

In [1]:
import pkgutil, torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

torch: 2.6.0+cu124
cuda available: True


## Imports and global configuration

In [7]:
import warnings
warnings.filterwarnings("ignore", message=".*Caching is incompatible with gradient checkpointing.*")
warnings.filterwarnings("ignore", message=".*Using the `WANDB_DISABLED` environment variable is deprecated.*")

from typing import List

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split

from datasets import Dataset
from trl import GRPOTrainer


from rbio.training.rewards import *

from rbio.training.utils import (
    set_random_seeds,
    load_mlp_classifier,
    setup_model_and_tokenizer,
    create_training_config,
    mlp_classifier_inference
)

from rbio.training.dual_utils import load_kg_data, add_dual, build_dual_prompt

# Training configuration
MODEL_NAME     = "Qwen/Qwen2.5-1.5B-Instruct"
N_STEPS        = 100
BATCH_SIZE     = 2
NUM_GENERATIONS = 2
SAVE_EVERY     = N_STEPS
EVAL_SAMPLES   = 50
N_RUNS         = 5
SEEDS          = [42, 43, 44, 45, 46]

MLP_MODEL_PATH = "./mlp_model.pt"
EMBEDDING_FILE = "./esm_embedding_dictionary_filled.pkl"
DATASET_PATHS  = ["./k562-train-v0.3.0.csv"]

_DUAL_FIELDS = [
    "gene_perturbed", "gene_monitored",
    "perturbed_gene_summary", "gene_monitored_rn_summaries", "potential_genes",
]

STEP_COUNT = 0

# Simplified Virtual Cell Model (VCM)

An MLP classifier trained to predict whether perturbing `gene_a` affects the expression of `gene_b`. Its output probability serves as the soft reward signal for LLM training.

In [ ]:
class MLPClassifier(nn.Module):
    """Simple MLP classifier for gene pair classification"""
    def __init__(self, input_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


mlp_model = None
embeddings_dict = None

# Dataset

`load_and_prepare_dataset` loads the raw CSV. `create_mlp_labeled_dataset_generator` labels examples on-the-fly using the MLP. `create_mlp_labeled_dual_dataset_generator` is a variant that also yields the KG-derived dual-task columns.

In [ ]:
def evaluate_on_holdout(model, tokenizer, test_df: pd.DataFrame, n: int = EVAL_SAMPLES) -> float:
    """Run greedy inference on n test rows and return accuracy vs MLP labels."""
    model.eval()
    correct = 0
    sample = test_df.head(n)
    for _, row in sample.iterrows():
        messages = [
            {"role": "system", "content": row["system_prompt"]},
            {"role": "user",   "content": row["user_prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        completion = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        predicted = extract_binary_answer(completion)
        if predicted is not None:
            predicted_str = "yes" if predicted else "no"
            true_str = row["classes"].split("|")[row["label"]]
            if predicted_str.lower() == true_str.lower():
                correct += 1
    model.train()
    return correct / len(sample)

## Load data, MLP, and create train/test split

The 80/20 split is fixed across all 10 runs so comparisons are fair.

In [ ]:
print("Loading dataset...")
dataset_df = load_and_prepare_dataset(DATASET_PATHS)

print("Loading MLP classifier...")
load_mlp_classifier(MLP_MODEL_PATH, EMBEDDING_FILE, MLPClassifier)

# Fixed train/test split shared across all runs
train_df, test_df = train_test_split(dataset_df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)} rows  |  Test: {len(test_df)} rows")

# Reward Functions

Same three-component reward as the base Rbio demo (format + mention + answer). The dual variant adds a fourth component: did the model correctly identify the monitored gene in the second generation?

In [ ]:
def reward_answer_against_label(completion: str, classes: str, class_confidence: str) -> float:
    answer = extract_binary_answer(completion)
    if answer is None:
        return 0.0
    answer = "yes" if answer else "no"
    possible_classes = classes.split("|")
    confidences = [float(c) for c in class_confidence.split("|")]
    for label, conf in zip(possible_classes, confidences):
        if answer == label.strip().lower():
            return conf
    return 0.0


def composite_formatting_reward(text: str, use_go: bool = False) -> float:
    at_least_one_think = has_at_least_one_think(text)
    has_tags = has_any_tag(text)
    checks = [
        at_least_one_think,
        low_untagged_ratio(text),
        is_not_too_long(text),
        has_one_answer(text),
        answer_after_thinks(text),
        thinks_have_text(text) * at_least_one_think,
        no_nested_tags(text) * has_tags,
        has_limited_thinks(text) * at_least_one_think,
        all_tags_properly_closed(text) * has_tags,
        ends_with_answer(text),
        starts_with_think(text),
    ]
    if use_go:
        checks = checks[:-1]
    return sum(checks) / len(checks)


def keywords_mentioned_in_think(text: str, keywords: str) -> float:
    keyword_list = [k for k in keywords.split("|") if k]
    if not keyword_list:
        return 1.0
    think_contents = extract_think(text)
    if not think_contents:
        return 0.0
    return sum(1 for kw in keyword_list if kw in think_contents) / len(keyword_list)


def compute_simple_reward(
    completions: List[str],
    label: List[int],
    classes: List[str],
    class_confidences: List[str],
    keywords: List[str],
    **kwargs
) -> List[float]:
    """Standard three-component reward (no dual task)."""
    global STEP_COUNT
    scores = []
    for completion, lbl, class_list, confidences, keyword_list in zip(
        completions, label, classes, class_confidences, keywords
    ):
        format_reward  = composite_formatting_reward(completion)
        mention_reward = keywords_mentioned_in_think(completion, keyword_list)
        answer_reward  = reward_answer_against_label(completion, class_list, confidences)
        scores.append(format_reward + 2.0 * answer_reward + mention_reward)
    STEP_COUNT += 1
    return scores

In [ ]:
def compute_dual_reward(
    completions: List[str],
    label: List[int],
    classes: List[str],
    class_confidences: List[str],
    keywords: List[str],
    second_completions: List[str] = None,
    gene_monitored_list: List[str] = None,
    **kwargs
) -> List[float]:
    """Four-component reward: format + mention + answer + dual gene identification."""
    global STEP_COUNT
    scores = []
    for i, (completion, lbl, class_list, confidences, keyword_list) in enumerate(zip(
        completions, label, classes, class_confidences, keywords
    )):
        format_reward  = composite_formatting_reward(completion)
        mention_reward = keywords_mentioned_in_think(completion, keyword_list)
        answer_reward  = reward_answer_against_label(completion, class_list, confidences)
        dual_reward = 0.0
        if second_completions and gene_monitored_list:
            gene_m = gene_monitored_list[i].upper()
            second = second_completions[i].upper()
            dual_reward = 1.0 if gene_m in second else 0.0
        scores.append(format_reward + 2.0 * answer_reward + mention_reward + dual_reward)
    STEP_COUNT += 1
    return scores


def make_rollout_fn(model, tokenizer, prompt_to_dual):
    """Return a rollout_func: generates first completions then dual completions."""
    import torch.nn.functional as F

    def rollout(prompts, trainer):
        n_gen = trainer.args.num_generations

        enc = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=256, do_sample=True, temperature=0.7,
                num_return_sequences=n_gen,
                return_dict_in_generate=True, output_scores=True,
            )

        prompt_len     = enc["input_ids"].shape[1]
        completion_ids = out.sequences[:, prompt_len:]
        first_texts    = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)

        logprobs = torch.stack(
            [F.log_softmax(s, dim=-1) for s in out.scores], dim=1
        )
        token_logprobs = logprobs.gather(2, completion_ids.unsqueeze(-1)).squeeze(-1)
        prompt_ids_rep = enc["input_ids"].repeat_interleave(n_gen, dim=0)

        prompts_rep = [p for p in prompts for _ in range(n_gen)]
        dual_prompts, gene_monitored_list = [], []
        for p, first_text in zip(prompts_rep, first_texts):
            dual = prompt_to_dual.get(p, {})
            answer = extract_binary_answer(first_text)
            answer_str = "yes" if answer is True else "no"
            dual_prompts.append(build_dual_prompt(
                dual.get("gene_perturbed", ""),
                answer_str,
                dual.get("perturbed_gene_summary", ""),
                dual.get("gene_monitored_rn_summaries", ""),
                dual.get("potential_genes", ""),
            ))
            gene_monitored_list.append(dual.get("gene_monitored", ""))

        enc2 = tokenizer(
            dual_prompts, return_tensors="pt", padding=True, truncation=True
        ).to(model.device)
        with torch.no_grad():
            out2 = model.generate(**enc2, max_new_tokens=64, do_sample=False)
        second_texts = tokenizer.batch_decode(
            out2[:, enc2["input_ids"].shape[1]:], skip_special_tokens=True
        )

        return {
            "prompt_ids":          prompt_ids_rep,
            "completion_ids":      completion_ids,
            "logprobs":            token_logprobs,
            "second_completions":  second_texts,
            "gene_monitored_list": gene_monitored_list,
        }
    return rollout

# Section A — Training WITHOUT Dual Task (5 runs)

Standard GRPO training using the three-component reward. Runs 5 times with seeds 42–46. After each run, the model is evaluated on the held-out test set and the training reward history is saved.

In [ ]:
results_no_dual = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"NO-DUAL run  |  seed={seed}")
    print(f"{'='*60}")

    set_random_seeds(seed)
    STEP_COUNT = 0

    model, tokenizer = setup_model_and_tokenizer(MODEL_NAME)

    dataset = Dataset.from_generator(
        create_mlp_labeled_dataset_generator,
        gen_kwargs={"dataset_df": train_df, "tokenizer": tokenizer, "balance_pos_neg": True},
    )
    config = create_training_config(
        output_dir=f"./checkpoints/no_dual_seed{seed}",
        batch_size=BATCH_SIZE,
        num_generations=NUM_GENERATIONS,
        max_steps=N_STEPS,
        save_every=SAVE_EVERY,
    )
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=compute_simple_reward,
        args=config,
        train_dataset=dataset,
    )
    trainer.train()

    print("Evaluating on held-out set...")
    accuracy = evaluate_on_holdout(model, tokenizer, test_df, n=EVAL_SAMPLES)
    print(f"Held-out accuracy: {accuracy:.3f}")

    results_no_dual.append({
        "seed":        seed,
        "log_history": trainer.state.log_history,
        "accuracy":    accuracy,
    })

    del trainer, model
    torch.cuda.empty_cache()
    print(f"Seed {seed} done.")

print("\nAll no-dual runs complete.")

# Section B — Training WITH Dual Task (5 runs)

Same GRPO setup but with an additional second model call per rollout. The model is first asked the binary perturbation question, then asked to identify the monitored gene from a KG-enriched candidate list. The dual reward is added to the total.

KG data is loaded **once** before the loop to avoid reloading 400 MB per seed.

In [ ]:
print("Loading KG data for dual task (loaded once for all 5 runs)...")
gs, kg = load_kg_data()

print("Enriching training split with dual-task columns...")
train_df_dual = add_dual(train_df, gs, kg)
print("Done.")

In [ ]:
results_dual = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"DUAL run  |  seed={seed}")
    print(f"{'='*60}")

    set_random_seeds(seed)
    STEP_COUNT = 0

    model, tokenizer = setup_model_and_tokenizer(MODEL_NAME)

    dataset_records = list(create_mlp_labeled_dual_dataset_generator(
        train_df_dual, tokenizer, balance_pos_neg=True
    ))
    dataset = Dataset.from_list(dataset_records)
    prompt_to_dual = {r["prompt"]: {k: r[k] for k in _DUAL_FIELDS} for r in dataset_records}

    config = create_training_config(
        output_dir=f"./checkpoints/dual_seed{seed}",
        batch_size=BATCH_SIZE,
        num_generations=NUM_GENERATIONS,
        max_steps=N_STEPS,
        save_every=SAVE_EVERY,
    )
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=compute_dual_reward,
        args=config,
        train_dataset=dataset,
        rollout_func=make_rollout_fn(model, tokenizer, prompt_to_dual),
    )
    trainer.train()

    print("Evaluating on held-out set...")
    accuracy = evaluate_on_holdout(model, tokenizer, test_df, n=EVAL_SAMPLES)
    print(f"Held-out accuracy: {accuracy:.3f}")

    results_dual.append({
        "seed":        seed,
        "log_history": trainer.state.log_history,
        "accuracy":    accuracy,
    })

    del trainer, model
    torch.cuda.empty_cache()
    print(f"Seed {seed} done.")

print("\nAll dual runs complete.")

# Results and Visualization

Collect reward curves and accuracy across all runs, then produce three seaborn plots saved as PNG files.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


def logs_to_df(results, condition):
    rows = []
    for r in results:
        for entry in r["log_history"]:
            if "reward" in entry:
                rows.append({
                    "step":      entry["step"],
                    "reward":    entry["reward"],
                    "seed":      r["seed"],
                    "condition": condition,
                })
    return pd.DataFrame(rows)


df_no_dual = logs_to_df(results_no_dual, "No Dual")
df_dual    = logs_to_df(results_dual,    "Dual")
df_rewards = pd.concat([df_no_dual, df_dual], ignore_index=True)

df_acc = pd.DataFrame(
    [{"condition": "No Dual", "seed": r["seed"], "accuracy": r["accuracy"]}
     for r in results_no_dual]
    + [{"condition": "Dual", "seed": r["seed"], "accuracy": r["accuracy"]}
       for r in results_dual]
)

print("Accuracy summary:")
print(df_acc.groupby("condition")["accuracy"].agg(["mean", "std"]).round(3))

In [ ]:
sns.set_theme(style="whitegrid", palette="Set2")

# Plot 1 — Reward curves over training steps
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(
    data=df_rewards, x="step", y="reward",
    hue="condition", errorbar="sd", ax=ax
)
ax.set_title("Training Reward over Steps (mean ± 1 SD, n=5 seeds)")
ax.set_xlabel("Training Step")
ax.set_ylabel("Mean Reward")
fig.tight_layout()
fig.savefig("plot_reward_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plot_reward_curves.png")

In [ ]:
# Plot 2 — Held-out accuracy comparison
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(
    data=df_acc, x="condition", y="accuracy",
    errorbar="sd", capsize=0.12, ax=ax
)
ax.set_title(f"Held-out Accuracy ({N_RUNS} runs, mean ± 1 SD)")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig("plot_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plot_accuracy.png")

In [ ]:
# Plot 3 — Reward distribution in the final 20% of training steps
max_step = df_rewards["step"].max()
final_mask = df_rewards["step"] >= max_step * 0.8
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(
    data=df_rewards[final_mask], x="condition", y="reward",
    ax=ax
)
ax.set_title("Reward Distribution — Final 20% of Training Steps")
ax.set_ylabel("Reward")
fig.tight_layout()
fig.savefig("plot_reward_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plot_reward_distribution.png")

In [ ]:
print("All plots exported:")
print("  plot_reward_curves.png      — reward over training steps")
print("  plot_accuracy.png           — held-out accuracy per condition")
print("  plot_reward_distribution.png— reward distribution (final steps)")

## Contact & Feedback

For issues or feedback about this tutorial please contact [virtualcellmodels@chanzuckerberg.com](mailto:virtualcellmodels@chanzuckerberg.com).

## Responsible Use

We are committed to advancing the responsible development and use of artificial intelligence.
Please follow our [Acceptable Use Policy](https://virtualcellmodels.cziscience.com/acceptable-use-policy) when engaging with our services.

For security or privacy issues, contact [security@chanzuckerberg.com](mailto:security@chanzuckerberg.com) or [privacy@chanzuckerberg.com](mailto:privacy@chanzuckerberg.com).